# DataForge pipeline worker — on Google Colab

For **testing only**, while the DGX is not reachable. Runs the same worker as
the DGX (AI check, hand tracking, dataset builds) against the **live app
server**: work comes from its Redis, clips from its MinIO, and results go back
to the app.

Colab joins your **Tailscale** network (no root needed) and forwards the VPS's
Redis (6379), MinIO (19000) and LiteLLM (4000) to its own `localhost`. Nothing
on the VPS is opened to the internet.

### Before you start (once)
1. **VPS on Tailscale.** On the VPS: `curl -fsSL https://tailscale.com/install.sh | sh && tailscale up`, then `tailscale ip -4` → the **VPS Tailscale IP**.
2. **Dokploy → Environment** (then Deploy):
   - `REDIS_BIND=<VPS Tailscale IP>` and `MINIO_BIND=<VPS Tailscale IP>`
   - `PIPELINE_HOURS=` (empty = always on) **while testing**, so the app sends work now, not at 22:00. Put `22:00-08:00` back afterwards.
3. **A Tailscale auth key** for Colab: https://login.tailscale.com/admin/settings/keys → *Generate auth key* → tick **Ephemeral** (Colab disappears from your network when the runtime stops).
4. **Colab Secrets** (🔑 icon on the left), with *Notebook access* on:
   | Name | Value |
   |---|---|
   | `GH_TOKEN` | a GitHub token that can read `RedMindRD/dataforge-pipeline` |
   | `TS_AUTHKEY` | the Tailscale auth key |
   | `REDIS_PASSWORD` | same as in Dokploy |
   | `PIPELINE_S3_SECRET_KEY` | same as in Dokploy |
   | `LLM_API_KEY` | the LiteLLM key (`EMPTY` if it has none) |
   | `HF_TOKEN` | optional — only to test **Publish** |

Then run the cells top to bottom. Runtime: **CPU is enough** (tracking is CPU; the model runs behind LiteLLM).

In [ ]:
#@title 1 · Settings
VPS_TS_IP = "100.x.y.z"  #@param {type:"string"}
BRANCH = "optimization"  #@param {type:"string"}
# LiteLLM as Colab sees it after the forward in step 4 (port 4000 on the VPS's Tailscale address).
LLM_API_BASE = "http://127.0.0.1:4000/v1"  #@param {type:"string"}
LLM_MODEL = "hosted_vllm/nvidia/Qwen3.6-35B-A3B-NVFP4"  #@param {type:"string"}
LLM_PORT_ON_VPS = 4000  #@param {type:"integer"}
WORKER_CONCURRENCY = 2  #@param {type:"integer"}

import sys
assert sys.version_info >= (3, 12), f"The pipeline needs Python 3.12+, this runtime has {sys.version.split()[0]}"
from google.colab import userdata
def secret(name, default=None):
    try:
        return userdata.get(name)
    except Exception:
        if default is None:
            raise RuntimeError(f"Add the Colab secret {name} (🔑 on the left) and give this notebook access")
        return default
print("Python", sys.version.split()[0], "— OK")

In [ ]:
#@title 2 · Get the pipeline code
import os, subprocess
os.chdir("/content")
if not os.path.isdir("/content/dataforge-pipeline"):
    url = f"https://{secret('GH_TOKEN')}@github.com/RedMindRD/dataforge-pipeline.git"
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "dataforge-pipeline"], check=True)
os.chdir("/content/dataforge-pipeline")
!git log --oneline -1

In [ ]:
#@title 3 · Install (5–10 min the first time)
!apt-get -qq update && apt-get -qq install -y ffmpeg libegl1 libgles2 socat > /dev/null
!pip -q install -e ".[ml]"
# Same fix as the Docker image: one headless OpenCV build that MediaPipe accepts.
!pip -q uninstall -y opencv-contrib-python opencv-python opencv-python-headless
!pip -q install "opencv-contrib-python-headless==5.0.0.93"
!python scripts/download_retarget_weights.py --dir models
!python -c "import cv2, mediapipe, mujoco, scipy, dfpipeline.worker; print('imports OK — cv2', cv2.__version__)"

In [ ]:
#@title 4 · Join Tailscale and forward Redis, MinIO and LiteLLM to localhost
import subprocess, time, os
!curl -fsSL https://tailscale.com/install.sh | sh > /dev/null
SOCK = "/tmp/tailscaled.sock"
# Userspace networking: no root network device needed, which is all Colab allows.
subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--state=mem:", f"--socket={SOCK}"],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
subprocess.run(["tailscale", f"--socket={SOCK}", "up", f"--authkey={secret('TS_AUTHKEY')}",
                "--hostname=colab-pipeline", "--accept-dns=false"], check=True)
# In userspace mode programs cannot open Tailscale addresses themselves, so each
# port is relayed: localhost:PORT → `tailscale nc` → VPS:PORT.
for local, remote in [(6379, 6379), (19000, 19000), (4000, LLM_PORT_ON_VPS)]:
    subprocess.Popen(["socat", f"TCP-LISTEN:{local},fork,reuseaddr",
                      f"EXEC:tailscale --socket={SOCK} nc {VPS_TS_IP} {remote}"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
!tailscale --socket=/tmp/tailscaled.sock status | head -5

In [ ]:
#@title 5 · Settings file + check every connection
env = {
    "APP_ENV": "production",
    "REDIS_URL": f"redis://:{secret('REDIS_PASSWORD')}@127.0.0.1:6379/0",
    "S3_ENDPOINT": "http://127.0.0.1:19000",
    "S3_ACCESS_KEY": "pipeline",
    "S3_SECRET_KEY": secret("PIPELINE_S3_SECRET_KEY"),
    "S3_BUCKET": "dataforge",
    "LLM_API_BASE": LLM_API_BASE,
    "LLM_API_KEY": secret("LLM_API_KEY", "EMPTY"),
    "LLM_MODEL": LLM_MODEL,
    "HF_TOKEN": secret("HF_TOKEN", ""),
    # Always on while testing; the app's PIPELINE_HOURS must be empty too.
    "PIPELINE_HOURS": "",
    "PIPELINE_TZ": "Asia/Kolkata",
    "NODE_NAME": "colab",
    "EXPORT_DATA_DIR": "/content/_datasets",
    "RETARGET_MODELS_DIR": "/content/dataforge-pipeline/models",
}
with open(".env.colab", "w") as f:
    f.writelines(f"{k}={v}\n" for k, v in env.items())
os.environ["ENV_FILE"] = ".env.colab"
print("wrote .env.colab")

import redis, minio, httpx
r = redis.Redis.from_url(env["REDIS_URL"], socket_timeout=10)
print("Redis :", "OK" if r.ping() else "no answer")
m = minio.Minio("127.0.0.1:19000", access_key="pipeline", secret_key=env["S3_SECRET_KEY"], secure=False)
print("MinIO :", "OK — bucket dataforge found" if m.bucket_exists("dataforge") else "bucket dataforge missing")
try:
    res = httpx.get(LLM_API_BASE.rstrip("/") + "/models", headers={"Authorization": f"Bearer {env['LLM_API_KEY']}"}, timeout=15)
    print("LLM   :", res.status_code, [x.get("id") for x in res.json().get("data", [])][:5])
except Exception as e:
    print("LLM   : not reachable —", e, "(AI checks will wait; tracking and builds still run)")

In [ ]:
#@title 6 · Run the worker (keeps running — stop it with ■)
# Same queues as the DGX. Upload a recording from the phone, then watch the
# checks arrive here; results show in the admin panel within seconds.
!ENV_FILE=.env.colab celery -A dfpipeline.worker worker -Q check,track,build -c {WORKER_CONCURRENCY} -n colab@%h --loglevel=INFO

### When you are done
- Stop the cell above, then **Runtime → Disconnect and delete runtime** (the ephemeral key removes Colab from Tailscale).
- In **Dokploy**, set `PIPELINE_HOURS=22:00-08:00` again and deploy.
- Work that was running when Colab stopped is not lost: it was never acknowledged, so Redis hands it out again to the next worker.